# ReLU and Leaky_ReLU Optimizations

The old implementation is shown below: 

In [ ]:
import numpy as np
class ReLU_old:

    def forward(self, inputs, training):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs < 0] = 0 
        

class Leaky_ReLU:
    def __init__(self, alpha = 0.01):
        self.alpha = alpha
    
    def forward(self, inputs, training):
        self.inputs = inputs
        self.output = cp.where(inputs > 0, inputs, self.alpha * inputs)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs < 0] *= self.alpha

class ReLU_new:

    def forward(self, inputs, training):
        self.output = np.maximum(0, inputs)
        return self.output
    
    def backward(self, dvalues): 
        self.dinputs = dvalues * (self.output > 0)
        return self.dinputs


## What's Wrong with the ReLU Implementation?

To start, there is major VRAM and time overhead of using any sort of direct copying, in our example this would be through the use of `.copy()`. 
* `self.inputs`: This line creates a object copy of the `inputs` parameter, which is used in the backwards pass to calculate the correct value for `self.dinputs`. 
    * The GPU memory controller is forced to allocate an entire new block of VRAM space matching the full layer layout. If inputs has shape $(N, H, W, C)$ then the GPU must copy this entire tensor across from VRAM onto the actual GPU. This casues **memory traffic latency**, where the compute units inside the GPU are stalled, as the computation is small but the memory overhead is large. 
*  `dvalues.copy()`: This case is extremely similar, yet we are still forced to maintain a copy of dvalues during training. Normally, in Python writing a line such as `variable = input_tensor` will have `variable` be a pointer that points to the memory address of the first value in `input_tensor`. 

A great example can be shown below, imagine we have a 4d input tensor, and a resulting architecture of the form: 

$$
{\Large\text{Input (1, 28, 28, 3)} \xrightarrow{\text{Convolution (1, 28, 28, 10)}} \,\ \xrightarrow{\text{ReLU (1, 28, 28 , 10)}} \,\ \xrightarrow{\text{Pooling (1, 14, 14, 10)}} } \ {\Large\text{Output (1, 14, 14, 10)}}
$$

Lets say we wrote the backwards pass of ReLU like so
```python
def backward(self, dvalues):
    dinputs = self.dvalues
    dinputs[self.inputs < 0] = 0 
```

Here, we have a pointer `dvalues` pointing to `Pooling.dinputs`, but now we have $\text{ReLU.dinputs} \rightarrow \text{ReLU.dvalues} \rightarrow \text{Pooling.dinputs}$. While this looks great for us, as we cut down on copying a 4d tensor, we still have the issue of updating during backpropagation. 
* Because we have a pointer pointing to `Pooling.dinputs`, when we perform the calculation for `ReLU.dinputs` we are updating the values in place, meaning we are corrupting the values of the pooling layer. 

The next question to ask is: Is there a way to avoid the computational (not space) cost of `.copy()`? 
In GPU programming, the answer is a resounding yes. We must perform an **out of place operation**, this will let us allocate a new array while doing our desired operation at the same time. Since we want to always keep a copy of `self.dinputs` for every layer, we'll attach the `self.` prefix to our `dinputs` as well. 

## Revamped ReLU Implementation

We'll start by avoiding any copy of our inputs, meaning we'll omit the line `self.inputs = inputs`, to get `self.output` we can do an out of place operation to get this output 
```python
def forward(self, inputs):
    self.output = xp.maximum(inputs, 0)
```

For the backward pass, we want to keep a copy of `dinputs`, so lets again to do an out of place operation and arrive at our output:

```python
def backward(self, dvalues):
    self.dinputs = dvalues * (self.outputs > 0)
```

## How can we Improve Leaky ReLU then? 

We can follow the same logic, we want to avoid any uncessary copies when we can, while also performing an out of place operation to create a new tensor, rather then performing a copy then an in place operation. 
The only difference between ReLU and Leaky_ReLU is the introduction of the $\alpha$ parameter. As a reminder, this is the resulting piecewise function:

$$\text{Leaky ReLU}(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha x & \text{if } x \le 0 \end{cases}$$

It could also be written as such 
$$
\text{Leaky ReLU}(x) = \text{max}(x, \alpha)
$$

We'll start with the forward pass. We want to use elementwise operations again for our cause, but exchange the $0$ with $\alpha$. This means, we'll first check if the value is less then zero, then if it is we can perform an operation where the input is multiplied by $\alpha$. 

### Why not use np or `cp.where`? 

If we execute a condition `xp.where(condition, x, y)`, we pass three arrays into the VRAM simultaneusly. Even if a position condition is `False`, the GPU still had to calculate, and pass the entire `x` array as an input argument to `xp.where()`. 

```python
self.output = xp.maximum(inputs, 0) + self.alpha * xp.minimum(inputs, 0)
```

In this case, we are performing a maximum check, a minimum check, and a multiplication operation at the same time. While this is an improvement, under the hood this presents a new problem. Under the hood, we end up with the scenario  
  
    Step 1: tmp1 = xp.maximum(0, inputs)  ──► Launches Kernel 1 (Allocates N elements)  
    Step 2: tmp2 = xp.minimum(0, inputs)  ──► Launches Kernel 2 (Allocates N elements)  
    Step 3: tmp3 = self.alpha * tmp2      ──► Launches Kernel 3 (Allocates N elements)  
    Step 4: output = tmp1 + tmp3          ──► Launches Kernel 4 (Allocates N elements)  

This means we launched 4 kernels when we want to perform this calculation in one kernel launch. To do this we can use the `cp.fuse()` decorator and combine these operations into one. 

    Step 1: Compiles expression into 1 Kernel  ──► Fuses all math into a single custom C++ CUDA block
    Step 2: Streams 'inputs' into registers    ──► Reads from VRAM exactly once
    Step 3: Computes Max, Min, Scale & Add     ──► Happens entirely within the GPU's cache
    Step 4: self.output = fused_result          ──► Writes to VRAM exactly 
    
```python
# Outside of any class
@cp.fuse()
def leaky_fused_relu(inputs, alpha):
    return (xp.maximum(inputs, 0) + alpha * xp.minimum(inputs, 0))

def _forward_gpu(self, inputs):
    self.output = leaky_fused_relu(inputs, self.alpha)
```
You may ask why the declarator is inside its own function, for this the declerator is unable to parse a class method because every method includes `self`. When the GPU tries to unpack `self` as an argument to a kernel, the GPU won't know what a python class instance is, this means we'll have to set it outside any class. 

## Leaky ReLU Backward Pass Implementation.

```python
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs < 0] *= self.alpha
```

Our goals are to get rid of the unecessary `copy()` and use an out of place operation to create a new tensor, speeding up computation. If we need to combine multiple elementwise operations we'll use `@cp.fuse()`to combine these operations. 

Lets first take a step back and write the formula for the Leaky ReLU backpropagation equation. 

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot \mathbb{I}(x > 0) + \alpha \frac{\partial L}{\partial y} \odot \mathbb{I}(x \le 0)
$$

Factoring out $\frac{\partial L}{\partial y}$ gives us
$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot ( \mathbb{I}(x > 0) + \alpha \mathbb{I}(x \le  0))
$$

Converting this directly to code results in 

```python
def backward(self, dvalues): 
    self.dinputs = dvalues * (self.output < 0) + dvalues * alpha * (self.output <= 0)
```

The problem again arises with 5 array allocations (1 main and 4 temporary). We can avoid this by doing some algebra on the indicator mask. 

We have two oposing conditions $(x > 0)$ and $(x <= 0)$. We can eliminate the second mask by using some algebra. 

Suppose we have a mask $M_{pos} = \mathbb{I}(x > 0)$, then plugging into our equation 

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot ( M_{pos} + \alpha \mathbb{I}(x \le  0))
$$

$\mathbb{I}(x \le  0)$ Is the same as writing $1 - M$. So we can rewrite the section as 
$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot ( M_{pos} + \alpha (1 - M_{pos}))
$$

We can distribute $\alpha$ and factor out $M$ and arrive at 
$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot (M (1 - \alpha) - \alpha)
$$

while this is good we can write it slightly better if we write thi|s using the negative mask instead. $M_{neg} = \mathbb{I}(x \le 0)$, meaning $M_{pos} = 1 - M_{neg}$. This will make us arrive at a final result 

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot (1 - M_{neg}(1-\alpha))
$$

Now, writing the final result will look as such 

```python
backward(self, dvalues):
    self.dinputs = dvalues * (1.0 - (self.output <= 0) * (1.0 - self.alpha))
```

The last thing to do is set this to use a declerator and we have our final code. This code can be found in `aether/blocks/activations.py`.